In [1]:
import pandas as pd
df = pd.read_csv("usarrests.csv")
df.head(10), df.shape, df.dtypes

(    Unnamed: 0  Murder  Assault  UrbanPop
 0      Alabama    13.2    236.0        58
 1       Alaska    10.0    263.0        48
 2      Arizona     8.1    294.0        80
 3     Arkansas     8.8    190.0        50
 4   California     9.0    276.0        91
 5     Colorado     7.9    204.0        78
 6  Connecticut     3.3    110.0        77
 7     Delaware     5.9    238.0        72
 8      Florida    15.4    335.0        80
 9      Georgia    17.4      NaN        60,
 (50, 4),
 Unnamed: 0        str
 Murder        float64
 Assault       float64
 UrbanPop        int64
 dtype: object)

## Fixing Column name
I am going to change the column name from unnamed to state, to make the tbales easier to read, and also makes coding easier down the line.

In [2]:
df = df.rename(columns={"Unnamed: 0": "State"})
## Adding a flagged column
df['Flagged'] = 'Clean'

In [3]:
df.head()

,State,Murder,Assault,UrbanPop,Flagged
0,Alabama,13.2,236.0,58,Clean
1,Alaska,10.0,263.0,48,Clean
2,Arizona,8.1,294.0,80,Clean
3,Arkansas,8.8,190.0,50,Clean
4,California,9.0,276.0,91,Clean


In [4]:
df.describe()

,Murder,Assault,UrbanPop
count,50.00000,49.000000,50.00000
mean,7.78800,182.183673,74.20000
std,4.35551,130.877435,73.40828
min,0.80000,45.000000,6.00000
25%,4.07500,109.000000,53.25000
50%,7.25000,159.000000,66.00000
75%,11.25000,249.000000,77.75000
max,17.40000,879.000000,570.00000


In [5]:
## Verifying why the max Urban population value is so high
df[df["UrbanPop"]>100]

,State,Murder,Assault,UrbanPop,Flagged
14,Iowa,2.2,56.0,570,Clean


### Flag 1 Impossible: Iowa Urban Population over 100%
Since Urban population cannot be more that total population, I made the assumption that there was a typo adding a 0 at the end. I am removing the 0 from Iowa's urban population, then verifying it has been updated.

In [6]:
## Flagging the line incase the value needs to be corrected in the future
df.at[14, "Flagged"] = "Impossible: Value was 570, replaced with 57"
## Replacing Iowa urban population with 57
df.loc[df["UrbanPop"]== 570, "UrbanPop"] = 57
## Verifying the change
df.loc[df["State"]== "Iowa"]

,State,Murder,Assault,UrbanPop,Flagged
14,Iowa,2.2,56.0,57,"Impossible: Value was 570, replaced with 57"


In [7]:
## checking for all null values
df.isna().sum()

State       0
Murder      0
Assault     1
UrbanPop    0
Flagged     0
dtype: int64

In [8]:
## verifying which row's assault value is null
df[df['Assault'].isna()]

,State,Murder,Assault,UrbanPop,Flagged
9,Georgia,17.4,NaN,60,Clean


### Flag 2 Impossible: Georgia's null value for assault
Georgia has no value inputed for its Assualts, so I am inputting the median value of assaults, then verifying the update.

In [9]:
## Flagging row
df.at[9, "Flagged"] = "Impossible: Georgia had null value for assault, replaced with median"
## creating a variable for the median assaults
median_assaults = df["Assault"].median()
## Filling in the missing value with the median variable
df["Assault"] = df["Assault"].fillna(median_assaults)
## verifying the value update
df.loc[df["State"]=="Georgia"]

,State,Murder,Assault,UrbanPop,Flagged
9,Georgia,17.4,159.0,60,Impossible: Georgia had null value for assault...


In [10]:
df[df["Murder"]<1]

,State,Murder,Assault,UrbanPop,Flagged
33,North Dakota,0.8,45.0,44,Clean


### Checking if North Dakota Murder Rate is implausible
To me North Dakota murder rate seems very low, I want to verify whether or not it should be flagged using IQR.

In [11]:
## Creating variables for quartiles 1 and 3
Q1 = df["Assault"].quantile(0.25)
Q3 = df["Assault"].quantile(0.75)

# Creating a variable for the IQR which is found by subtracting Q1 from Q3
IQR = Q3 - Q1

# Determining the lower boundary 
lower = Q1 - (1.5 * IQR)

# Comparing North Dakota Murder Rate vs the lower Outlier Threshold
print(f"{df["Assault"].min()}")
print(f"Lower Outlier Threshold: {lower}")

45.0
Lower Outlier Threshold: -101.0


North Dakota murder rate is still within the threshold, so it will not be flagged.

### Looking into the max assault value
I noticed there is a large jump from the Q3 and the max value in Assaults column. I wanted to take a closer look to see what the cause of it is.

In [12]:
df[df["Assault"] > 249].sort_values(by="Assault", ascending=False)

,State,Murder,Assault,UrbanPop,Flagged
39,South Carolina,14.4,879.0,48,Clean
32,North Carolina,13.0,337.0,45,Clean
8,Florida,15.4,335.0,80,Clean
19,Maryland,11.3,300.0,67,Clean
2,Arizona,8.1,294.0,80,Clean
30,New Mexico,11.4,285.0,70,Clean
4,California,9.0,276.0,91,Clean
1,Alaska,10.0,263.0,48,Clean
23,Mississippi,16.1,259.0,44,Clean
21,Michigan,12.1,255.0,74,Clean


In [13]:
# Determining the upper boundary 
upper = Q3 + (1.5 * IQR)

# Comparing North Dakota Murder Rate vs the lower Outlier Threshold
print(f"{df["Assault"].max()}")
print(f"Upper Outlier Threshold: {upper}")

879.0
Upper Outlier Threshold: 459.0


In [14]:
## Flagging the row
df.at[39, "Flagged"] = "Extreme: South Carolina assault rate is out of outlier threshold, no changes made."

### Flag 3 Extreme: South Carolina's Large assault rate
South Carolina has an extremely large assault rate, nearly double the upper threshold, but since it also has a fairly large murder rate I am going to assume it is just an outlier and not a typo. I am flagging it as extreme because it deviates so much from the rest of the data, so it is worth looking into later to verify that it is accurate data.

In [15]:
df.describe()

,Murder,Assault,UrbanPop
count,50.00000,50.000000,50.000000
mean,7.78800,181.720000,63.940000
std,4.35551,129.576554,16.453286
min,0.80000,45.000000,6.000000
25%,4.07500,109.000000,53.250000
50%,7.25000,159.000000,66.000000
75%,11.25000,249.000000,76.500000
max,17.40000,879.000000,91.000000


In [16]:
df[df["UrbanPop"]<10]

,State,Murder,Assault,UrbanPop,Flagged
31,New York,11.1,254.0,6,Clean


### Flag 4 Impossible: New York's urban population too low
New york is known to have a large urban population, so I know that 6 as the urban population was a typo. To fix it I replaced it with the median of urban population and then verified the change.

In [17]:
## Flagging row
df.at[31, "Flagged"] = "Impossible: New Yorks uban population had a value of 6. Replaced with median"
median_urbanPop = df["UrbanPop"].median()
df.loc[df["UrbanPop"]== 6, "UrbanPop"] = median_urbanPop
df.loc[df["State"] == "New York"]

,State,Murder,Assault,UrbanPop,Flagged
31,New York,11.1,254.0,66,Impossible: New Yorks uban population had a va...


In [18]:
df.describe()

,Murder,Assault,UrbanPop
count,50.00000,50.000000,50.000000
mean,7.78800,181.720000,65.140000
std,4.35551,129.576554,14.170982
min,0.80000,45.000000,32.000000
25%,4.07500,109.000000,54.500000
50%,7.25000,159.000000,66.000000
75%,11.25000,249.000000,76.500000
max,17.40000,879.000000,91.000000


In [19]:
df[df["Flagged"] != "Clean"]

,State,Murder,Assault,UrbanPop,Flagged
9,Georgia,17.4,159.0,60,Impossible: Georgia had null value for assault...
14,Iowa,2.2,56.0,57,"Impossible: Value was 570, replaced with 57"
31,New York,11.1,254.0,66,Impossible: New Yorks uban population had a va...
39,South Carolina,14.4,879.0,48,Extreme: South Carolina assault rate is out of...


## Flagged Values
- I Changed the unnamed column to "State"
- Row 14, Iowa's Urban population is 570%, it is impossible for the urban population to be 570% of the total population, it must be below 100%. I assumed that 570 was a typo, where a 0 was accidentally added, so I replaced the 570 with 57.
- Row 39, South Carolina was marked 879 assaults per 100,000 people. I would mark that as extreme, it is over 500 assualts more than the next highest number, making it improbable, but not impossible, since it is top 5 for murders. I kept that data point the same because the high murder rate shows its possible for the state to have an extremely high assault rate.
- Row 9, Georgia has nothing in the assualt column which would be impossible given they have the highest murder rate. I replaced Georgia's empty assualt value with the median value of assualts.
- Row 31, New York's Urban population has a value of 6% which is impossible because New York is known for having one of the largest metropolitian cities in the USA. To fix this I will replaced the 6% with the median Urban Population percent.

## Cleaning Log
- Changed unnamed column to State
- Replaced Urban population from 570 to 57
- Replaced Georgia's null value for assaults to median with a value of 159
- Replaced New Yorks Urban population to median with a value of 66

### Github URL